# Q5 Failure Tests
## CSCE 5306 — Distributed Systems, Project 3

---

## How This Notebook Works

This notebook is split into **two independent phases** that are intentionally kept separate.
This mirrors the standard workflow for empirical systems work: collect raw data first,
then render/analyze it separately so you can iterate on presentation without re-running experiments.

```
Phase 1 — Test Execution          Phase 2 — Rendering
─────────────────────────         ─────────────────────────────
Run Docker cluster                Read raw log files
Perform failure action        →   Format as terminal-styled PNGs
Capture docker logs               Save to docs/report/screenshots/
Write to docs/report/logs/        (referenced by report.tex)
```

### Phase 1: Test Execution
- Manages the Docker cluster (`make up` / `make down` between each TC)
- Performs the failure action for each test case
- Saves raw log text to `docs/report/logs/tc{N}_raw.txt`
- **Requires Docker Desktop to be running**
- Run the five TC sections in order (TC1 → TC5)
- If a TC behaves unexpectedly, consult `docs/spec/06_failure_test_matrix.md`
  for the manual Docker commands

### Phase 2: Rendering
- Reads from `docs/report/logs/tc{N}_raw.txt`
- Produces terminal-styled PNGs in `docs/report/screenshots/`
- **Does not require Docker** — can be re-run at any time to tweak styling
  without touching the cluster
- Each PNG is referenced by `report.tex` via `\\includegraphics{screenshots/tcN_*.png}`

### Typical full workflow
```
1. Run the Shared Helpers cell
2. Run Phase 1 cells (TC1 → TC5) — requires Docker
3. Run Phase 2 cells — produces PNGs
4. Fill in \\todo{} placeholders in report.tex with observed behaviour
5. make pdf   (from repo root)
```

### Re-rendering only (no cluster needed)
```
1. Run the Shared Helpers cell
2. Skip Phase 1 entirely (logs already exist in docs/report/logs/)
3. Run Phase 2 cells
```

---
## Shared Helpers
_Run this cell first — both phases depend on it._

In [ ]:
"""
Shared helpers used by both Phase 1 (execution) and Phase 2 (rendering).

Paths are resolved relative to the repo root, so this notebook can be opened
from any working directory.  The repo root is found by walking upward until
AGENTS.md is located.
"""
import os
import re
import subprocess
import time
import textwrap
from pathlib import Path

import matplotlib
matplotlib.use("Agg")   # headless — works without a display server
import matplotlib.pyplot as plt


# ── Repo-root detection ──────────────────────────────────────────────────────

def _find_repo_root() -> Path:
    """Walk upward from cwd until AGENTS.md is found."""
    for candidate in [Path(os.getcwd()).resolve(), *Path(os.getcwd()).resolve().parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    raise RuntimeError("Cannot locate repo root — AGENTS.md not found.")

REPO_ROOT    = _find_repo_root()
COMPOSE_FILE = REPO_ROOT / "server" / "docker-compose.yml"
LOGS_DIR     = REPO_ROOT / "docs" / "report" / "logs"
SCREENSHOTS  = REPO_ROOT / "docs" / "report" / "screenshots"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
SCREENSHOTS.mkdir(parents=True, exist_ok=True)

print(f"Repo root  : {REPO_ROOT}")
print(f"Logs dir   : {LOGS_DIR}")
print(f"Screenshots: {SCREENSHOTS}")


# ── Shell / Docker helpers (Phase 1) ────────────────────────────────────────

COMPOSE_CMD = ["docker", "compose", "-f", str(COMPOSE_FILE)]
ALL_NODES   = ["fishing1", "fishing2", "fishing3", "fishing4", "fishing5", "fishing6"]


def run(cmd: list[str], *, check: bool = True, capture: bool = False,
        timeout: int = 120) -> subprocess.CompletedProcess:
    """Run a subprocess command, optionally capturing stdout/stderr."""
    result = subprocess.run(cmd, capture_output=capture, text=True,
                            timeout=timeout, check=False)
    if check and result.returncode != 0:
        print(f"[WARN] {' '.join(cmd)} exited {result.returncode}")
        if result.stderr:
            print(result.stderr[:400])
    return result


def cluster_up(wait: int = 10) -> None:
    """Build and start the 6-node cluster, then pause for leader election."""
    print("Starting cluster…")
    run(COMPOSE_CMD + ["up", "--build", "-d"])
    print(f"Waiting {wait}s for leader election…")
    time.sleep(wait)


def cluster_down() -> None:
    """Tear down all cluster containers."""
    print("Stopping cluster…")
    run(COMPOSE_CMD + ["down"], check=False)


def docker_logs(service: str, tail: int = 80) -> str:
    """Return the last *tail* lines from a container's stdout + stderr."""
    r = run(["docker", "logs", "--tail", str(tail), service],
            capture=True, check=False)
    return (r.stdout + r.stderr).strip()


def collect_logs(services: list[str], tail: int = 40) -> str:
    """Concatenate recent logs from *services* with clear node headers."""
    parts = []
    for svc in services:
        text = docker_logs(svc, tail=tail)
        if text:
            parts.append(f"{'='*6} {svc} {'='*6}")
            parts.append(text)
    return "\n".join(parts)


def find_leader(nodes: list[str] | None = None) -> str | None:
    """Return the service name of the most recent leader, or None."""
    for svc in (nodes or ALL_NODES):
        if "became leader" in docker_logs(svc, tail=200):
            return svc
    return None


def save_raw_log(tc: int, content: str) -> Path:
    """Write raw log text to docs/report/logs/tc{N}_raw.txt and return the path."""
    path = LOGS_DIR / f"tc{tc}_raw.txt"
    path.write_text(content, encoding="utf-8")
    print(f"Raw log saved → {path}  ({len(content.splitlines())} lines)")
    return path


# ── Screenshot renderer (Phase 2) ────────────────────────────────────────────

def render_screenshot(tc: int, title: str, max_lines: int = 60) -> Path:
    """
    Read docs/report/logs/tc{N}_raw.txt, render it as a terminal-styled PNG,
    and save to docs/report/screenshots/tc{N}_{slug}.png.

    The file name encodes the TC number so report.tex can \\includegraphics it directly.
    Returns the output path.
    """
    raw_path = LOGS_DIR / f"tc{tc}_raw.txt"
    if not raw_path.exists():
        raise FileNotFoundError(
            f"{raw_path} not found.  Run Phase 1 (TC{tc}) first."
        )

    log_text = raw_path.read_text(encoding="utf-8")

    # Word-wrap long lines and truncate if needed
    raw_lines = log_text.splitlines()
    wrapped: list[str] = []
    for line in raw_lines:
        wrapped.extend(textwrap.wrap(line, width=110) or [""])
    if len(wrapped) > max_lines:
        omitted = len(wrapped) - max_lines
        wrapped = [f"[… {omitted} earlier lines omitted …]"] + wrapped[-max_lines:]

    # Build figure
    fig_h = max(3.5, 0.21 * len(wrapped) + 1.0)
    fig, ax = plt.subplots(figsize=(14, fig_h))
    fig.patch.set_facecolor("#1e1e1e")
    ax.set_facecolor("#1e1e1e")
    ax.axis("off")

    # Title bar
    fig.text(0.5, 0.99, title, ha="center", va="top",
             fontsize=9.5, fontweight="bold", color="#ffffff",
             fontfamily="monospace")

    # Log body
    ax.text(0.01, 0.97, "\n".join(wrapped),
            transform=ax.transAxes, va="top", ha="left",
            fontsize=7.5, fontfamily="monospace",
            color="#d4d4d4", linespacing=1.4)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    # Safe slug: replace non-alphanumeric runs with underscores
    slug = re.sub(r'[^a-z0-9]+', '_', title.lower())[:40].strip('_')
    out_path = SCREENSHOTS / f"tc{tc}_{slug}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"Screenshot saved → {out_path}")
    return out_path


print("Helpers loaded.  Ready for Phase 1 (execution) and Phase 2 (rendering).")

---
---
# PHASE 1 — Test Execution

> **Requires Docker Desktop to be running.**  
> Run TC1 → TC5 in order. Each section resets the cluster before running its scenario.
> Raw log output is written to `docs/report/logs/` — these files are the durable
> intermediate artifacts between Phase 1 and Phase 2.
>
> If you have already run Phase 1 and only want to re-render the screenshots,
> skip this entire section and go directly to **Phase 2**.

---
### TC1 — Leader Crash and Re-Election

**Scenario**: Kill the current leader container and observe that the remaining followers
detect the missing heartbeat, elect a new leader, and resume normal operation.

**Expected**: One follower transitions to candidate within 1.5–3 s, wins a majority vote,
and begins sending AppendEntries heartbeats as the new leader.

In [ ]:
# ── TC1 Setup ────────────────────────────────────────────────────────────────
cluster_down()
cluster_up(wait=10)

leader_tc1 = find_leader()
print(f"Initial leader: {leader_tc1}")

In [ ]:
# ── TC1 Action ───────────────────────────────────────────────────────────────
assert leader_tc1, "No leader found — check cluster logs via collect_logs(ALL_NODES)"

print(f"Stopping leader: {leader_tc1}")
run(["docker", "stop", leader_tc1])

print("Waiting 10s for re-election (max election timeout = 3s + margin)…")
time.sleep(10)

survivors = [s for s in ALL_NODES if s != leader_tc1]
new_leader = find_leader(nodes=survivors)
print(f"New leader: {new_leader}")

In [ ]:
# ── TC1 Log Capture ──────────────────────────────────────────────────────────
header = (
    f"TC1: Leader Crash and Re-Election\n"
    f"Killed: {leader_tc1}  |  New leader: {new_leader}\n"
    f"{'─'*70}\n"
)
raw_tc1 = header + collect_logs(survivors, tail=45)
save_raw_log(1, raw_tc1)

---
### TC2 — Follower Crash and Recovery

**Scenario**: Kill a follower, let the cluster commit entries while it is down,
then restart the follower and verify it catches up via the leader's heartbeat.

**Expected**: Cluster continues committing entries with the remaining 5 nodes (majority
of 6 intact). Restarted follower receives full log on next heartbeat and applies
committed entries.

In [ ]:
# ── TC2 Setup ────────────────────────────────────────────────────────────────
cluster_down()
cluster_up(wait=10)

leader_tc2 = find_leader()
follower_tc2 = next(s for s in ALL_NODES if s != leader_tc2)
print(f"Leader: {leader_tc2}  |  Will stop follower: {follower_tc2}")

In [ ]:
# ── TC2 Action ───────────────────────────────────────────────────────────────
print(f"Stopping {follower_tc2}…")
run(["docker", "stop", follower_tc2])

print("Waiting 6s (3 heartbeat cycles) while follower is down…")
time.sleep(6)
logs_while_down = collect_logs([leader_tc2], tail=25)

print(f"Restarting {follower_tc2}…")
run(["docker", "start", follower_tc2])

print("Waiting 5s for catch-up via AppendEntries heartbeat…")
time.sleep(5)

In [ ]:
# ── TC2 Log Capture ──────────────────────────────────────────────────────────
header = (
    f"TC2: Follower Crash and Recovery\n"
    f"Stopped follower: {follower_tc2}  |  Leader: {leader_tc2}\n"
    f"{'─'*70}\n"
    f"--- Leader logs while follower was down ---\n"
)
logs_after = collect_logs([leader_tc2, follower_tc2], tail=30)
raw_tc2 = header + logs_while_down + "\n\n--- After follower restart ---\n" + logs_after
save_raw_log(2, raw_tc2)

---
### TC3 — Network Partition (Pause / Unpause)

**Scenario**: Pause a container to simulate a network partition (process is alive but
unreachable), then unpause it to simulate recovery.

**Expected**: If the paused node is the leader, the remaining nodes re-elect a new leader.
On unpause the node receives a heartbeat, discovers the current term, and resynchronises.
A stale leader steps down upon seeing a higher term.

In [ ]:
# ── TC3 Setup ────────────────────────────────────────────────────────────────
cluster_down()
cluster_up(wait=10)

leader_tc3 = find_leader()
# Pause the leader for the most interesting behaviour (triggers re-election)
pause_target = leader_tc3 or "fishing2"
print(f"Leader: {leader_tc3}  |  Will pause: {pause_target}")

In [ ]:
# ── TC3 Action ───────────────────────────────────────────────────────────────
print(f"Pausing {pause_target}…")
run(["docker", "pause", pause_target])

print("Waiting 9s for re-election among surviving nodes…")
time.sleep(9)
survivors_tc3 = [s for s in ALL_NODES if s != pause_target]
new_leader_tc3 = find_leader(nodes=survivors_tc3)
logs_paused = collect_logs(survivors_tc3[:3], tail=25)

print(f"Unpausing {pause_target}…")
run(["docker", "unpause", pause_target])

print("Waiting 5s for resynchronisation…")
time.sleep(5)

In [ ]:
# ── TC3 Log Capture ──────────────────────────────────────────────────────────
header = (
    f"TC3: Network Partition (pause/unpause)\n"
    f"Paused: {pause_target}  |  New leader during partition: {new_leader_tc3}\n"
    f"{'─'*70}\n"
    f"--- Logs from surviving nodes during partition ---\n"
)
logs_after_unpause = collect_logs([pause_target] + survivors_tc3[:2], tail=25)
raw_tc3 = header + logs_paused + "\n\n--- After unpause (resynchronisation) ---\n" + logs_after_unpause
save_raw_log(3, raw_tc3)

---
### TC4 — New Node Joining the Cluster

**Scenario**: Start with only 5 nodes running, build up log history, then bring the
6th node online and verify it integrates as a follower.

**Expected**: `fishing6` starts as a follower, receives the full log from the leader
on the next heartbeat, applies all committed entries, and its state is consistent
with the rest of the cluster.

In [ ]:
# ── TC4 Setup ────────────────────────────────────────────────────────────────
cluster_down()

print("Starting 5-node cluster (fishing1–fishing5 only)…")
run(COMPOSE_CMD + ["up", "--build", "-d",
                   "fishing1", "fishing2", "fishing3", "fishing4", "fishing5"])
print("Waiting 10s for leader election…")
time.sleep(10)

leader_tc4 = find_leader(nodes=["fishing1","fishing2","fishing3","fishing4","fishing5"])
print(f"Leader (5-node cluster): {leader_tc4}")

In [ ]:
# ── TC4 Action ───────────────────────────────────────────────────────────────
print("Waiting 6s to accumulate heartbeat log entries on the 5-node cluster…")
time.sleep(6)

print("Starting fishing6…")
run(COMPOSE_CMD + ["up", "-d", "fishing6"])

print("Waiting 5s for fishing6 to receive AppendEntries and sync…")
time.sleep(5)

In [ ]:
# ── TC4 Log Capture ──────────────────────────────────────────────────────────
logs_new_node  = docker_logs("fishing6", tail=40)
logs_leader_tc4 = docker_logs(leader_tc4 or "fishing1", tail=25)

header = (
    f"TC4: New Node Joining the Cluster\n"
    f"New node: fishing6  |  Leader: {leader_tc4}\n"
    f"{'─'*70}\n"
)
raw_tc4 = (
    header
    + "=== fishing6 (new node — logs after joining) ===\n" + logs_new_node
    + "\n\n=== " + (leader_tc4 or "fishing1") + " (leader) ===\n" + logs_leader_tc4
)
save_raw_log(4, raw_tc4)

---
### TC5 — Split Vote and Election Retry

**Scenario**: Kill the leader and one follower simultaneously to increase the probability
of a split vote among the remaining four nodes.

**Expected**: Multiple candidates may start elections in the same term; if no candidate
gets a majority (3 of 4 needed) the term ends without a leader; a new election starts
with fresh random timeouts and an incremented term; eventually a leader wins.

> **Note:** A genuine split vote is non-deterministic.  If it does not occur, the logs
> will still demonstrate the mechanism: different election timeouts per node and
> multiple terms before a leader is chosen.  Both outcomes satisfy the assignment.

In [ ]:
# ── TC5 Setup ────────────────────────────────────────────────────────────────
cluster_down()
cluster_up(wait=10)

leader_tc5 = find_leader()
followers_tc5 = [s for s in ALL_NODES if s != leader_tc5]
kill_follower = followers_tc5[0]

print(f"Leader: {leader_tc5}  |  Will also kill: {kill_follower}")

In [ ]:
# ── TC5 Action ───────────────────────────────────────────────────────────────
print(f"Stopping {leader_tc5} and {kill_follower} simultaneously…")
run(["docker", "stop", leader_tc5, kill_follower])

print("Waiting 14s for election(s) to complete (may require multiple rounds)…")
time.sleep(14)

survivors_tc5 = [s for s in ALL_NODES if s not in {leader_tc5, kill_follower}]
new_leader_tc5 = find_leader(nodes=survivors_tc5)
print(f"Final leader: {new_leader_tc5}")

In [ ]:
# ── TC5 Log Capture ──────────────────────────────────────────────────────────
header = (
    f"TC5: Split Vote and Election Retry\n"
    f"Killed: {leader_tc5}, {kill_follower}  |  Final leader: {new_leader_tc5}\n"
    f"{'─'*70}\n"
)
raw_tc5 = header + collect_logs(survivors_tc5, tail=50)
save_raw_log(5, raw_tc5)

# Tear down after the last test case
cluster_down()
print("\nPhase 1 complete. All raw logs saved to docs/report/logs/.")
print("Proceed to Phase 2 to generate screenshots.")

---
---
# PHASE 2 — Rendering

> **Does not require Docker.**  
> Reads from `docs/report/logs/tc{N}_raw.txt` and produces terminal-styled PNGs
> in `docs/report/screenshots/`.
>
> You can run this section independently at any time — for example to adjust
> screenshot styling, change the number of lines shown, or regenerate after editing
> the raw log files manually.
>
> The output filenames are referenced directly by `report.tex`:
> `\\includegraphics{screenshots/tc{N}_*.png}`

In [ ]:
# ── Render all five test case screenshots ────────────────────────────────────
#
# render_screenshot(tc, title) reads docs/report/logs/tc{tc}_raw.txt and
# writes docs/report/screenshots/tc{tc}_{slug}.png.
#
# Edit 'title' strings here to change the caption shown at the top of each PNG.
# Edit 'max_lines' to control how many log lines appear before truncation.

renders = [
    (1, "TC1: Leader Crash and Re-Election",         60),
    (2, "TC2: Follower Crash and Recovery",          60),
    (3, "TC3: Network Partition (pause/unpause)",    60),
    (4, "TC4: New Node Joining the Cluster",         60),
    (5, "TC5: Split Vote and Election Retry",        60),
]

generated = []
for tc, title, max_lines in renders:
    try:
        path = render_screenshot(tc, title, max_lines=max_lines)
        generated.append(path)
    except FileNotFoundError as e:
        print(f"[SKIP] {e}")

print(f"\nPhase 2 complete. {len(generated)}/5 screenshots generated.")

---
## Verification

Run the cell below to confirm both the raw logs and the final PNGs are present,
then check `report.tex` for any remaining `\\todo{}` placeholders before building the PDF.

In [ ]:
print("Raw log files (docs/report/logs/):")
for p in sorted(LOGS_DIR.glob("tc*.txt")):
    lines = len(p.read_text(encoding="utf-8").splitlines())
    print(f"  {p.name:30s}  {lines} lines")

print()
print("Screenshot PNGs (docs/report/screenshots/):")
for p in sorted(SCREENSHOTS.glob("tc*.png")):
    size_kb = p.stat().st_size // 1024
    print(f"  {p.name:45s}  {size_kb} KB")

print()
print("Next step: fill in \\todo{} placeholders in docs/report/report.tex,")
print("then run `make pdf` from the repo root.")